In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Ejercicio 1

- $n$ observaciones $\mathbf x_i\in\mathbb R^p$, **sin etiquetas**.
- Una asignación $C:\{1,\dots,n\}\to\{1,\dots,K\}$ y centroides $\boldsymbol\mu_1,\dots,\boldsymbol\mu_K\in\mathbb R^p$.

**Función de distorsión**

$$
J(C,\boldsymbol\mu)=\sum_{i=1}^n\|\mathbf x_i-\boldsymbol\mu_{C(i)}\|_2^2 .
$$

**Algoritmo** (descenso por coordenadas sobre $J$). A partir de centroides iniciales $\boldsymbol\mu^{(0)}_1,\dots,\boldsymbol\mu^{(0)}_K$, repetir:

1. **Asignación** (fijos los centroides, minimizamos en $C$):
$$
C^{(t+1)}(i)=\arg\min_j\|\mathbf x_i-\boldsymbol\mu^{(t)}_j\|_2^2 .
$$
2. **Actualización** (fija la asignación, minimizamos en $\boldsymbol\mu$):
$$
\boldsymbol\mu^{(t+1)}_j=\frac{\sum_i\mathbf 1\{C^{(t+1)}(i)=j\}\,\mathbf x_i}{\sum_i\mathbf 1\{C^{(t+1)}(i)=j\}} .
$$

**Criterio de parada:** la asignación no cambia, o el desplazamiento máximo de los centroides $\max_j\|\boldsymbol\mu^{(t+1)}_j-\boldsymbol\mu^{(t)}_j\|_2$ es menor que `tol`. Si se alcanza `max_iter` sin cumplirlo, se avisa con `warnings.warn`.

**Inicialización:** elegimos $K$ observaciones distintas al azar como centroides iniciales. Como el resultado depende de esa elección (ítem c), el parámetro `n_init` repite el ajuste con `n_init` inicializaciones distintas y conserva la de menor $J$.

**Clusters vacíos:** si en algún paso ningún punto queda asignado al cluster $j$, el promedio no está definido. Como ningún $C(i)$ vale $j$, $\boldsymbol\mu_j$ no aparece en ningún término de $J$: cualquier valor que le demos **no cambia $J$** en ese paso, y el descenso se mantiene. Lo que cambia es qué pasa en las iteraciones siguientes. Implementamos tres estrategias (parámetro `vacios`):

- `"mantener"`: conserva el centroide anterior. Es lo más simple. Si los demás centroides se alejan, puede volver a captar puntos, pero si no, queda "muerto" y en la práctica terminamos con $K-1$ clusters.
- `"aleatorio"`: lo reubica en una observación elegida al azar. En la asignación siguiente, ese punto queda a distancia $0$ de un centroide, así que $J$ baja al menos lo que aportaba ese punto.
- `"lejano"`: lo reubica en la observación con mayor $\|\mathbf x_i-\boldsymbol\mu_{C(i)}\|_2^2$, es decir, la peor representada. Es la que garantiza la mayor reducción de $J$ en la asignación siguiente. Es la estrategia de `KMeans` de scikit-learn, que además corrige el centroide del cluster que cede el punto.

Para el paso de asignación conviene calcular de una vez la matriz $D\in\mathbb R^{n\times K}$ con $D_{ij}=\|\mathbf x_i-\boldsymbol\mu_j\|_2^2$ usando *broadcasting*:
`X[:, None, :] - centroides[None, :, :]` tiene forma `(n, K, p)`.

In [ ]:
class myKMeans:
    """
    Algoritmo de K-medias.
    """
    def __init__(self, K, max_iter=300, tol=1e-6, n_init=1, vacios="mantener", random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - K: número de clusters
        - max_iter: número máximo de iteraciones de cada corrida
        - tol: tolerancia sobre el desplazamiento máximo de los centroides
        - n_init: cantidad de inicializaciones (se conserva la de menor J)
        - vacios: estrategia para los clusters vacíos: "mantener", "aleatorio" o "lejano"
        - random_state: semilla para la inicialización
        """

        if vacios not in ("mantener", "aleatorio", "lejano"):
            raise ValueError(f"Estrategia desconocida para clusters vacíos: {vacios}")

        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.vacios = vacios
        self.random_state = random_state

        # Resultados del ajuste
        self.centroides = None  # centroides finales (K, p)
        self.etiquetas = None  # asignación final C(i) (n,)
        self.J = None  # distorsión final (la menor entre las n_init corridas)
        self.J_hist = None  # distorsión en cada iteración de la mejor corrida
        self.n_iter_ = None  # iteraciones de la mejor corrida

    def asignar(self, X, centroides):
        """
        Paso de asignación: C(i) = argmin_j ||x_i − μ_j||².

        Parámetros:
        - X: datos (n, p)
        - centroides: (K, p)

        Retorna:
        - etiquetas: (n,)
        """

        # TODO:
        # - Calcular la matriz de distancias al cuadrado D (n, K)
        # - Devolver el índice del centroide más cercano a cada x_i

        # Opción 1: muestra a muestra
        D = np.zeros((X.shape[0], self.K))  # (n, K)
        for i in range(...):
            for j in range(...):
                D[i, j] = ...

        # Opción 2: centroide a centroide
        D = np.zeros((X.shape[0], self.K))  # (n, K)
        for j in range(...):
            D[:, j] = ...  # (n,)

        # Opción 3: broadcasting (n, 1, p) - (1, K, p) -> (n, K, p)
        D = np.sum((... - ...) ** 2, axis=...)  # (n, K)

        return np.argmin(D, axis=...)

    def actualizar(self, X, etiquetas, centroides, rng=None):
        """
        Paso de actualización: μ_j = promedio de los x_i con C(i) = j.

        Si un cluster queda vacío, su centroide se define según self.vacios:
        - "mantener": conserva el centroide anterior
        - "aleatorio": lo reubica en una observación elegida al azar
        - "lejano": lo reubica en la observación más lejana a su centroide

        Parámetros:
        - X: datos (n, p)
        - etiquetas: asignación C(i) (n,)
        - centroides: centroides de la iteración anterior (K, p)
        - rng: generador de números aleatorios (se usa con "aleatorio")

        Retorna:
        - centroides_nuevos: (K, p)
        """

        # TODO:
        # - Promediar las observaciones de cada cluster no vacío
        # - Guardar los índices de los clusters vacíos
        # - Reubicar los centroides vacíos según la estrategia elegida

        centroides_nuevos = centroides.copy()
        vacios = []  # índices de los clusters vacíos

        for j in range(self.K):
            Xj = X[...]
            if len(Xj) > 0:
                centroides_nuevos[j] = ...
            else:
                vacios.append(j)

        if len(vacios) == 0 or self.vacios == "mantener":
            return centroides_nuevos

        if self.vacios == "aleatorio":
            indices = rng.choice(..., size=..., replace=False)

        elif self.vacios == "lejano":
            d = ... # (n,)
            indices = ...

        centroides_nuevos[vacios] = X[indices]

        return centroides_nuevos

    def distorsion(self, X, etiquetas, centroides):
        """
        J(C, μ) = Σ_i ||x_i − μ_{C(i)}||²
        """

        # Opción 1: muestra a muestra
        J = 0
        for i in range(...):
            j = ...
            J += ...

        # Opción 2: centroide a centroide
        J = 0
        for j in range(...):
            Xj = ...
            J += ...

        # Opción 3: vectorizada
        J = ...

        return J

    def fit(self, X):
        """
        Ajusta el modelo n_init veces y conserva la corrida de menor distorsión.

        Parámetros:
        - X: (n, p)

        Retorna:
        - self
        """

        # TODO:
        # - Repetir n_init veces (con el mismo generador rng):
        #   - inicializar con K observaciones elegidas al azar, sin reposición
        #   - alternar asignación y actualización, guardando J en cada iteración
        #   - parar si la asignación no cambia o el desplazamiento máximo es menor que tol;
        #     si se llega a max_iter, avisar con warnings.warn
        # - Conservar la corrida de menor J

        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        n = X.shape[0]

        self.J = np.inf
        for _ in range(self.n_init):
            indices = rng.choice(..., size=..., replace=False)
            centroides = X[indices]  # (K, p)
            etiquetas = None
            J_hist = []

            for t in range(self.max_iter):
                etiquetas_nuevas = ...  # 1. asignación, (n,)
                centroides_nuevos = ...  # 2. actualización, (K, p)
                J_hist.append(...)

                max_desp = ...
                mismas_etiquetas = ...
                etiquetas, centroides = etiquetas_nuevas, centroides_nuevos
                if ...:
                    break
            else:
                warnings.warn(f"myKMeans no convergió en {self.max_iter} iteraciones")

            if ...:  # la mejor corrida hasta ahora
                self.centroides = centroides
                self.etiquetas = etiquetas
                self.J_hist = J_hist
                self.J = J_hist[-1]
                self.n_iter_ = t + 1

        return self

    def predict(self, X):
        """
        Asigna cada observación al centroide más cercano.

        Parámetros:
        - X: (m, p)

        Retorna:
        - etiquetas: (m,)
        """

        return ...

##### Solución

In [ ]:
class myKMeans:
    """
    Algoritmo de K-medias.
    """
    def __init__(self, K, max_iter=300, tol=1e-6, n_init=1, vacios="mantener", random_state=None):
        """
        Inicializa el modelo.

        Parámetros:
        - K: número de clusters
        - max_iter: número máximo de iteraciones de cada corrida
        - tol: tolerancia sobre el desplazamiento máximo de los centroides
        - n_init: cantidad de inicializaciones (se conserva la de menor J)
        - vacios: estrategia para los clusters vacíos: "mantener", "aleatorio" o "lejano"
        - random_state: semilla para la inicialización
        """

        if vacios not in ("mantener", "aleatorio", "lejano"):
            raise ValueError(f"Estrategia desconocida para clusters vacíos: {vacios}")

        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.vacios = vacios
        self.random_state = random_state

        # Resultados del ajuste
        self.centroides = None  # centroides finales (K, p)
        self.etiquetas = None  # asignación final C(i) (n,)
        self.J = None  # distorsión final (la menor entre las n_init corridas)
        self.J_hist = None  # distorsión en cada iteración de la mejor corrida
        self.n_iter_ = None  # iteraciones de la mejor corrida

    def asignar(self, X, centroides):
        """
        Paso de asignación: C(i) = argmin_j ||x_i − μ_j||².

        Parámetros:
        - X: datos (n, p)
        - centroides: (K, p)

        Retorna:
        - etiquetas: (n,)
        """

        # Opción 1: muestra a muestra
        D = np.zeros((X.shape[0], self.K))  # (n, K)
        for i in range(X.shape[0]):
            for j in range(self.K):
                D[i, j] = np.sum((X[i] - centroides[j]) ** 2)

        # Opción 2: centroide a centroide
        D = np.zeros((X.shape[0], self.K))  # (n, K)
        for j in range(self.K):
            D[:, j] = np.sum((X - centroides[j]) ** 2, axis=1)  # (n,)

        # Opción 3: broadcasting (n, 1, p) - (1, K, p) -> (n, K, p)
        D = np.sum((X[:, None, :] - centroides[None, :, :]) ** 2, axis=2)  # (n, K)

        return np.argmin(D, axis=1)

    def actualizar(self, X, etiquetas, centroides, rng=None):
        """
        Paso de actualización: μ_j = promedio de los x_i con C(i) = j.

        Si un cluster queda vacío, su centroide se define según self.vacios:
        - "mantener": conserva el centroide anterior
        - "aleatorio": lo reubica en una observación elegida al azar
        - "lejano": lo reubica en la observación más lejana a su centroide

        Parámetros:
        - X: datos (n, p)
        - etiquetas: asignación C(i) (n,)
        - centroides: centroides de la iteración anterior (K, p)
        - rng: generador de números aleatorios (se usa con "aleatorio")

        Retorna:
        - centroides_nuevos: (K, p)
        """

        centroides_nuevos = centroides.copy()
        vacios = []  # índices de los clusters vacíos

        for j in range(self.K):
            Xj = X[etiquetas == j]
            if len(Xj) > 0:
                centroides_nuevos[j] = np.mean(Xj, axis=0)
            else:
                vacios.append(j)

        if len(vacios) == 0 or self.vacios == "mantener":
            return centroides_nuevos

        if self.vacios == "aleatorio":
            indices = rng.choice(X.shape[0], size=len(vacios), replace=False)

        elif self.vacios == "lejano":
            # Distancia al cuadrado de cada x_i a su centroide (ningún x_i está en un cluster vacío)
            d = np.sum((X - centroides_nuevos[etiquetas]) ** 2, axis=1)  # (n,)
            indices = np.argsort(d)[::-1][:len(vacios)]  # las más lejanas

        centroides_nuevos[vacios] = X[indices]

        return centroides_nuevos

    def distorsion(self, X, etiquetas, centroides):
        """
        J(C, μ) = Σ_i ||x_i − μ_{C(i)}||²
        """

        # Opción 1: muestra a muestra
        J = 0
        for i in range(X.shape[0]):
            j = etiquetas[i]
            J += np.sum((X[i] - centroides[j]) ** 2)

        # Opción 2: centroide a centroide
        J = 0
        for j in range(self.K):
            Xj = X[etiquetas == j]
            J += np.sum((Xj - centroides[j]) ** 2)

        # Opción 3: vectorizada
        J = np.sum((X - centroides[etiquetas]) ** 2)

        return J

    def fit(self, X):
        """
        Ajusta el modelo n_init veces y conserva la corrida de menor distorsión.

        Parámetros:
        - X: (n, p)

        Retorna:
        - self
        """

        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=float)
        n = X.shape[0]

        self.J = np.inf
        for _ in range(self.n_init):
            indices = rng.choice(n, size=self.K, replace=False)
            centroides = X[indices]  # (K, p)
            etiquetas = None
            J_hist = []

            for t in range(self.max_iter):
                etiquetas_nuevas = self.asignar(X, centroides)  # 1. asignación, (n,)
                centroides_nuevos = self.actualizar(X, etiquetas_nuevas, centroides, rng)  # 2. actualización, (K, p)
                J_hist.append(self.distorsion(X, etiquetas_nuevas, centroides_nuevos))

                max_desp = np.max(np.linalg.norm(centroides_nuevos - centroides, axis=1))
                mismas_etiquetas = np.array_equal(etiquetas_nuevas, etiquetas)
                etiquetas, centroides = etiquetas_nuevas, centroides_nuevos
                if mismas_etiquetas or max_desp < self.tol:
                    break
            else:
                warnings.warn(f"myKMeans no convergió en {self.max_iter} iteraciones")

            if J_hist[-1] < self.J:  # la mejor corrida hasta ahora
                self.centroides = centroides
                self.etiquetas = etiquetas
                self.J_hist = J_hist
                self.J = J_hist[-1]
                self.n_iter_ = t + 1

        return self

    def predict(self, X):
        """
        Asigna cada observación al centroide más cercano.

        Parámetros:
        - X: (m, p)

        Retorna:
        - etiquetas: (m,)
        """

        return self.asignar(X, self.centroides)

##### Función auxiliar para graficar

Una función auxiliar para graficar cualquier agrupamiento en $\mathbb R^2$ (la reutilizamos en los tres ejercicios). Las observaciones con etiqueta $-1$ (ruido, en DBSCAN) se grafican con una cruz negra.

In [ ]:
COLORES = plt.cm.tab10.colors

def graficar_clusters(X, etiquetas, ax, centroides=None, titulo=""):
    """
    Grafica un agrupamiento en R^2.

    Parámetros:
    - X: datos (n_samples, 2)
    - etiquetas: (n,), con -1 para el ruido
    - ax: eje de matplotlib
    - centroides: (K, 2) o None
    """
    etiquetas = np.asarray(etiquetas)
    ruido = etiquetas == -1

    colores = [COLORES[e % len(COLORES)] for e in etiquetas[~ruido]]
    ax.scatter(X[~ruido, 0], X[~ruido, 1], c=colores, s=10)

    if ruido.any():
        ax.scatter(X[ruido, 0], X[ruido, 1], c="k", marker="x", s=15, linewidths=0.8, label="ruido")
        ax.legend(loc="best", fontsize=8)

    if centroides is not None:
        ax.scatter(centroides[:, 0], centroides[:, 1], c="k", marker="X", s=150, edgecolors="w", linewidths=1.5)

    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")

#### a) Verificación con `make_blobs`

Probamos dos conjuntos (3 y 5 centros) con distintos valores de $K$: el correcto, uno menor y uno mayor.

In [ ]:
from sklearn.datasets import make_blobs

X_3, y_3 = make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=42)
X_5, y_5 = make_blobs(n_samples=500, centers=5, cluster_std=1.0, random_state=7)

casos = [
    (X_3, 3, 2),
    (X_3, 3, 3),
    (X_3, 3, 5),
    (X_5, 5, 3),
    (X_5, 5, 5),
    (X_5, 5, 7),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (X, centers, K) in zip(axes.ravel(), casos):
    km = myKMeans(K=K, random_state=0).fit(X)
    graficar_clusters(
        X,
        km.etiquetas,
        ax,
        km.centroides,
        titulo=f"centers={centers}, K={K} | J={km.J:.1f}, {km.n_iter_} iter.",
    )
plt.tight_layout()
plt.show()

Por construcción (actividad conceptual 1), $J$ no puede crecer de una iteración a la siguiente. Lo verificamos con la historia `J_hist` de una corrida:

In [ ]:
km = myKMeans(K=5, random_state=3).fit(X_5)

print("J en cada iteración:", np.round(km.J_hist, 2))
print("¿J es no creciente?", np.all(np.diff(km.J_hist) <= 1e-9))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, km.n_iter_ + 1), km.J_hist, "o-")
ax.set_xlabel("iteración t")
ax.set_ylabel("J")
ax.set_title("Distorsión a lo largo de las iteraciones")
plt.show()

Comparamos con `KMeans` de scikit-learn. Su atributo `inertia_` es exactamente nuestra $J$. Para que la comparación sea justa, le pasamos como `init` los **mismos centroides iniciales** que elige nuestra implementación. Por defecto, scikit-learn usa `k-means++`, una inicialización más elaborada que la aleatoria.

In [ ]:
from sklearn.cluster import KMeans

for X, K in [(X_3, 3), (X_5, 5)]:
    km = myKMeans(K=K, random_state=0).fit(X)

    # Mismos centroides iniciales que myKMeans (mismo generador y misma semilla)
    idx0 = np.random.default_rng(0).choice(X.shape[0], size=K, replace=False)
    km_sk = KMeans(n_clusters=K, init=X[idx0], n_init=1).fit(X)

    print(
        f"K={K}: J (myKMeans) = {km.J:.4f} | inertia_ (sklearn) = {km_sk.inertia_:.4f} | "
        f"centroides iguales: {np.allclose(np.sort(km.centroides, axis=0), np.sort(km_sk.cluster_centers_, axis=0))}",
    )

#### b) Método del codo

Usamos un conjunto con 4 centros. Como el resultado depende de la inicialización (ver ítem c), para cada $K$ nos quedamos con la menor $J$ entre 10 corridas.

In [ ]:
X_4, y_4 = make_blobs(n_samples=400, centers=4, cluster_std=0.60, random_state=0)

Ks = np.arange(1, 11)
J_K = []
for K in Ks:
    J_K.append(min(myKMeans(K=K, random_state=s).fit(X_4).J for s in range(10)))
J_K = np.array(J_K)

for K, J in zip(Ks, J_K):
    print(f"K = {K}: J = {J:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(X_4[:, 0], X_4[:, 1], s=10, c="gray")
axes[0].set_title("Datos (4 centros)")
axes[1].plot(Ks, J_K, "o-")
axes[1].axvline(4, color="r", ls="--", lw=1, label="K = 4")
axes[1].set_xticks(Ks)
axes[1].set_xlabel("K")
axes[1].set_ylabel("J")
axes[1].set_title("Método del codo")
axes[1].legend()
plt.tight_layout()
plt.show()

**Valor óptimo: $K=4$.** Hasta $K=4$, cada cluster nuevo separa dos grupos reales y $J$ cae mucho (de 3730 a 280). A partir de $K=4$, la curva se aplana. Agregar un centroide solo parte un grupo que ya estaba bien representado, y la mejora es marginal (de 280 a 250 al pasar a $K=5$). El "codo" es el punto donde cambia la pendiente.

**¿Por qué no elegir el $K$ que minimiza $J$?** Porque $J^*(K)=\min_{C,\boldsymbol\mu}J$ es **no creciente** en $K$:

- Si tomamos la solución óptima con $K$ clusters y agregamos un centroide nuevo ubicado sobre cualquier observación $\mathbf x_i$, la distancia de $\mathbf x_i$ a su centroide pasa a ser $0$ y la de los demás puntos no aumenta. Entonces $J^*(K+1)\le J^*(K)$.
- En el extremo $K=n$, cada observación es su propio centroide y $J=0$.

Minimizar $J$ siempre lleva a $K=n$: tantos clusters como datos, que no resume nada. $J$ mide qué tan bien los centroides **representan** a los datos, no si los grupos son reales. Por eso buscamos el punto a partir del cual agregar clusters deja de mejorar $J$ de forma apreciable.

#### c) Dependencia de la inicialización

Fijamos $K=4$ sobre los mismos datos del ítem anterior y repetimos el ajuste con 200 semillas distintas.

In [ ]:
K = 4
n_corridas = 200

modelos_c = [myKMeans(K=K, random_state=s).fit(X_4) for s in range(n_corridas)]
J_corridas = np.array([m.J for m in modelos_c])

print(f"J final: mínimo {J_corridas.min():.2f}, máximo {J_corridas.max():.2f}, media {J_corridas.mean():.2f}, desvío {J_corridas.std():.2f}")

J_min = J_corridas.min()
exito = np.isclose(J_corridas, J_min, rtol=1e-3)
print(f"\nCorridas que alcanzan el mínimo: {exito.sum()} de {n_corridas} ({exito.mean():.1%})")

valores, cuentas = np.unique(J_corridas.round(1), return_counts=True)
print("\nValores finales de J (redondeados) y frecuencia:")
for v, c in zip(valores, cuentas):
    print(f"  J = {v:.1f}: {c} corridas")

In [ ]:
mejor = modelos_c[np.argmin(J_corridas)]
peor = modelos_c[np.argmax(J_corridas)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].hist(J_corridas, bins=40)
axes[0].set_xlabel("J final")
axes[0].set_ylabel("frecuencia")
axes[0].set_title(f"{n_corridas} corridas con K={K}")
graficar_clusters(X_4, mejor.etiquetas, axes[1], mejor.centroides, titulo=f"Menor J = {mejor.J:.1f}")
graficar_clusters(X_4, peor.etiquetas, axes[2], peor.centroides, titulo=f"Mayor J = {peor.J:.1f}")
plt.tight_layout()
plt.show()

Todas las corridas convergen (el algoritmo siempre se detiene), pero **no** siempre al mismo punto:

- Alrededor de 3 de cada 4 corridas llegan a $J\approx280$, que corresponde a los 4 grupos reales.
- El resto queda atrapado en soluciones con $J\approx700$. Allí, dos grupos reales comparten un centroide y otro grupo queda partido en dos. Es un **punto fijo** del algoritmo: ningún paso de asignación o actualización lo mejora, aunque $J$ sea más del doble del mínimo.

Es lo que vimos en la actividad conceptual 2: como $J$ no es convexa, el descenso por coordenadas solo garantiza llegar a un mínimo local. La solución práctica es repetir el ajuste con varias inicializaciones y quedarse con la de menor $J$ (parámetro `n_init`).

Si una corrida aislada falla con probabilidad $q$, la probabilidad de que fallen las `n_init` corridas independientes es $q^{\,n\_init}$. Con $q\approx0.25$ y `n_init=10`, eso es del orden de $10^{-6}$.

Con `n_init`, cada ajuste prueba varias inicializaciones y se queda con la de menor $J$:

In [ ]:
J_ninit = np.array([myKMeans(K=4, n_init=10, random_state=s).fit(X_4).J for s in range(100)])

print(f"100 ajustes con n_init=10: J mínimo = {J_ninit.min():.2f}, J máximo = {J_ninit.max():.2f}")
print(f"Ajustes que alcanzan el mínimo: {np.isclose(J_ninit, J_min, rtol=1e-3).sum()} de 100")

#### d) Clusters no esféricos

Los datos se generan con la función `generar_X_elip`, con semillas fijas: la guía 3.3 usa la misma función y obtiene exactamente los mismos datos, sin depender de un archivo.

In [ ]:
def generar_X_elip():
    """
    Cuatro grupos gaussianos deformados por una transformación lineal (los datos de la guía).

    Retorna:
    - X_elip: (400, 2)
    - y_true: grupo de cada observación (400,)
    """
    X_raw, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.60, random_state=0)
    rng = np.random.RandomState(13)
    return X_raw @ rng.normal(size=(2, 2)), y_true


X_elip, y_true = generar_X_elip()

km_elip = myKMeans(K=4, n_init=10, random_state=0).fit(X_elip)

# Distorsión de la partición verdadera (cada grupo con su propia media)
mu_true = np.array([X_elip[y_true == k].mean(axis=0) for k in range(4)])
J_true = km_elip.distorsion(X_elip, y_true, mu_true)

print(f"J de la solución de K-medias: {km_elip.J:.2f}")
print(f"J de la partición verdadera: {J_true:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
graficar_clusters(X_elip, y_true, axes[0], mu_true, titulo=f"Grupos verdaderos (J = {J_true:.1f})")
graficar_clusters(X_elip, km_elip.etiquetas, axes[1], km_elip.centroides, titulo=f"myKMeans, K=4 (J = {km_elip.J:.1f})")
for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

**Para discutir:** ¿Por qué K-medias no recupera los grupos elongados, aun con `n_init=10`?

#### e) *(Adicional)* Clusters vacíos

Con datos continuos, los clusters vacíos casi no aparecen. Son más frecuentes cuando hay **observaciones repetidas**: la inicialización puede elegir dos puntos idénticos como centroides y, ante el empate, `np.argmin` asigna todos esos puntos al de menor índice, así que el otro cluster queda vacío.

Para provocarlo, redondeamos los datos de 3 centros a enteros y usamos $K=8$. Comparamos las tres estrategias con la misma inicialización y, después, sobre 200 inicializaciones distintas.

In [ ]:
# Datos redondeados a enteros: muchas observaciones repetidas
X_red = np.round(X_3)
print(f"Observaciones distintas: {len(np.unique(X_red, axis=0))} de {len(X_red)}\n")

estrategias = ["mantener", "aleatorio", "lejano"]

# Misma inicialización (semilla 33) con las tres estrategias
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, vacios in zip(axes, estrategias):
    km = myKMeans(K=8, vacios=vacios, random_state=33).fit(X_red)
    n_no_vacios = len(np.unique(km.etiquetas))
    graficar_clusters(
        X_red,
        km.etiquetas,
        ax,
        km.centroides,
        titulo=f'vacios="{vacios}": {n_no_vacios} clusters no vacíos, J={km.J:.1f}',
    )
    print(f"{vacios}: J_hist = {np.round(km.J_hist, 1)}")
plt.tight_layout()
plt.show()

# 200 inicializaciones distintas
filas = []
for vacios in estrategias:
    modelos = [myKMeans(K=8, vacios=vacios, random_state=s).fit(X_red) for s in range(200)]
    J_s = np.array([m.J for m in modelos])
    con_vacios = sum(len(np.unique(m.etiquetas)) < 8 for m in modelos)
    filas.append({"estrategia": vacios, "J medio": J_s.mean(), "J mínimo": J_s.min(), "corridas con algún cluster vacío al final": con_vacios})

pd.DataFrame(filas).round(1)

## Ejercicio 2

- Hiperparámetros: el radio $\varepsilon>0$ y la cantidad mínima de puntos $\nu$.
- Vecindario de $\mathbf x_i$: $N_\varepsilon(\mathbf x_i)=\{\mathbf x_j:\ d(\mathbf x_i,\mathbf x_j)<\varepsilon\}$. **Incluye a $\mathbf x_i$**.
- $\mathbf x_i$ es **objeto núcleo** si $|N_\varepsilon(\mathbf x_i)|\ge\nu$.
- **Frontera:** no es núcleo, pero está en el vecindario de algún núcleo. Se asigna a ese cluster.
- **Ruido:** no está en el vecindario de ningún núcleo. No se asigna.

**Convención de etiquetas durante el algoritmo**

| etiqueta | significado |
|---|---|
| $-2$ | no visitada |
| $-1$ | ruido (o frontera todavía no alcanzada por ningún núcleo) |
| $c\ge0$ | asignada al cluster $c$ |

Al terminar, todas las etiquetas son $\ge -1$, y las que quedaron en $-1$ son ruido.

**`expandir_cluster`** recorre una lista de puntos pendientes, que empieza con los vecinos de un núcleo:

- vecino con etiqueta $-1$ → pasa al cluster actual (es frontera, **no** expande);
- vecino con etiqueta $-2$ → pasa al cluster actual; si además es núcleo, sus vecinos se agregan a la lista;
- vecino ya asignado a un cluster → no se hace nada.

In [ ]:
class myDBSCAN:
    """
    Implementación del algoritmo DBSCAN.
    """
    def __init__(self, eps=0.5, nu=5):
        """
        Inicializa el modelo.

        Parámetros:
        - eps : float
            Radio del vecindario
        - nu : int
            Cantidad mínima de puntos (incluido el propio punto) para ser objeto núcleo
        """

        self.eps = eps
        self.nu = nu

        self.X = None
        self.etiquetas = None  # (n,)  -2: no visitada, -1: ruido, c >= 0: cluster
        self.es_nucleo = None  # (n,)  True si la observación es objeto núcleo
        self.n_clusters = None

    def obtener_vecinos(self, i):
        """
        Índices de las observaciones a distancia < eps de x_i (incluye a i).
        """

        # TODO:
        # - Calcular la distancia euclídea de x_i a todas las observaciones
        # - Devolver (como list) los índices con distancia < eps

        d = ...

        return list(...)

    def expandir_cluster(self, i, vecinos, c):
        """
        Expande el cluster c a partir del objeto núcleo x_i y su lista de vecinos.
        """

        # TODO:
        # - Asignar x_i al cluster c
        # - Recorrer la lista de pendientes con pop(0):
        #     etiqueta -1 -> frontera: asignar c
        #     etiqueta -2 -> asignar c; si es núcleo, extend con sus vecinos

        self.etiquetas[i] = c
        pendientes = list(vecinos)

        while pendientes:
            j = pendientes.pop(0)

            if self.etiquetas[j] == -1:
                ...

            elif self.etiquetas[j] == -2:
                ...

    def fit(self, X):
        """
        Aplica DBSCAN sobre X.

        Parámetros:
        - X: datos (n, p)

        Retorna:
        - etiquetas: (n,), con -1 para el ruido
        """

        # TODO:
        # - Inicializar todas las etiquetas en -2
        # - Para cada observación no visitada: calcular su vecindario
        #     menos de nu puntos -> etiqueta -1
        #     al menos nu puntos -> núcleo: expandir_cluster con el índice actual y luego incrementarlo

        self.X = np.asarray(X, dtype=float)
        n = self.X.shape[0]

        self.etiquetas = np.full(n, -2)
        self.es_nucleo = np.zeros(n, dtype=bool)

        c = 0
        for i in range(n):
            ...

        self.n_clusters = c

        return self.etiquetas

##### Solución

In [ ]:
class myDBSCAN:
    """
    Implementación del algoritmo DBSCAN.
    """
    def __init__(self, eps=0.5, nu=5):
        """
        Inicializa el modelo.

        Parámetros:
        - eps : float
            Radio del vecindario
        - nu : int
            Cantidad mínima de puntos (incluido el propio punto) para ser objeto núcleo
        """

        self.eps = eps
        self.nu = nu

        self.X = None
        self.etiquetas = None  # (n,)  -2: no visitada, -1: ruido, c >= 0: cluster
        self.es_nucleo = None  # (n,)  True si la observación es objeto núcleo
        self.n_clusters = None

    def obtener_vecinos(self, i):
        """
        Índices de las observaciones a distancia < eps de x_i (incluye a i).
        """

        d = np.linalg.norm(self.X - self.X[i], axis=1)  # (n,)

        return list(np.where(d < self.eps)[0])

    def expandir_cluster(self, i, vecinos, c):
        """
        Expande el cluster c a partir del objeto núcleo x_i y su lista de vecinos.
        """

        self.etiquetas[i] = c
        pendientes = list(vecinos)

        while pendientes:
            j = pendientes.pop(0)

            if self.etiquetas[j] == -1:
                # Ya visitado y no es núcleo: es frontera de este cluster (no expande)
                self.etiquetas[j] = c

            elif self.etiquetas[j] == -2:
                # No visitado: pertenece al cluster; si es núcleo, seguimos expandiendo
                self.etiquetas[j] = c
                vecinos_j = self.obtener_vecinos(j)
                if len(vecinos_j) >= self.nu:
                    self.es_nucleo[j] = True
                    pendientes.extend(vecinos_j)

            # Si ya tenía un cluster asignado, no hacemos nada

    def fit(self, X):
        """
        Aplica DBSCAN sobre X.

        Parámetros:
        - X: datos (n, p)

        Retorna:
        - etiquetas: (n,), con -1 para el ruido
        """

        self.X = np.asarray(X, dtype=float)
        n = self.X.shape[0]

        self.etiquetas = np.full(n, -2)
        self.es_nucleo = np.zeros(n, dtype=bool)

        c = 0
        for i in range(n):
            if self.etiquetas[i] != -2:  # ya visitado
                continue

            vecinos = self.obtener_vecinos(i)
            if len(vecinos) < self.nu:
                self.etiquetas[i] = -1  # ruido (por ahora: puede ser frontera más adelante)
            else:
                self.es_nucleo[i] = True
                self.expandir_cluster(i, vecinos, c)
                c += 1

        self.n_clusters = c

        return self.etiquetas

#### a) Verificación con `make_moons`

In [ ]:
from sklearn.datasets import make_moons

X_m, y_m = make_moons(n_samples=400, noise=0.1, random_state=0)

eps_values = [0.05, 0.1, 0.15, 0.2, 0.3]
nu_values = [3, 5, 10]

resultados_db = {}
filas = []
for eps in eps_values:
    for nu in nu_values:
        db = myDBSCAN(eps=eps, nu=nu)
        etiquetas = db.fit(X_m)
        resultados_db[(eps, nu)] = etiquetas
        filas.append({"eps": eps, "nu": nu, "clusters": db.n_clusters, "ruido": np.sum(etiquetas == -1), "núcleos": db.es_nucleo.sum()})

pd.DataFrame(filas)

In [ ]:
fig, axes = plt.subplots(len(nu_values), len(eps_values), figsize=(20, 11), sharex=True, sharey=True)
for f, nu in enumerate(nu_values):
    for col, eps in enumerate(eps_values):
        et = resultados_db[(eps, nu)]
        graficar_clusters(
            X_m,
            et,
            axes[f, col],
            titulo=f"ε={eps}, ν={nu}: {et.max() + 1} clusters, {np.sum(et == -1)} ruido",
        )
plt.tight_layout()
plt.show()

#### b) *(Adicional)* Comparación con `DBSCAN` de scikit-learn

En `sklearn.cluster.DBSCAN`:

- `min_samples` es la cantidad de puntos en el vecindario **contando al propio punto**: la misma convención que nuestro $\nu$. Entonces `min_samples = nu` (no `nu - 1`).
- El vecindario usa $d\le\varepsilon$, y nosotros usamos $d<\varepsilon$. Con datos continuos, la probabilidad de que una distancia sea exactamente $\varepsilon$ es nula, así que no hay diferencia práctica. Sí importa con datos en una grilla, como en la actividad conceptual 3.
- El ruido se etiqueta con $-1$, igual que en nuestra implementación. Los índices de los núcleos están en `core_sample_indices_`.

In [ ]:
from sklearn.cluster import DBSCAN

filas = []
for eps in eps_values:
    for nu in nu_values:
        db = myDBSCAN(eps=eps, nu=nu)
        et = db.fit(X_m)
        sk = DBSCAN(eps=eps, min_samples=nu).fit(X_m)
        nucleos_sk = np.zeros(len(X_m), dtype=bool)
        nucleos_sk[sk.core_sample_indices_] = True
        filas.append({
            "eps": eps,
            "nu": nu,
            "clusters": et.max() + 1,
            "ruido": np.sum(et == -1),
            "clusters sklearn": sk.labels_.max() + 1,
            "ruido sklearn": np.sum(sk.labels_ == -1),
            "etiquetas iguales": np.array_equal(et, sk.labels_),
            "núcleos iguales": np.array_equal(db.es_nucleo, nucleos_sk),
        })

pd.DataFrame(filas)

## Ejercicio 3 *(Adicional)*

Generamos los cinco conjuntos con $n=500$ y los **estandarizamos** (media 0 y varianza 1 en cada coordenada) para que los valores de $\varepsilon$ estén en una escala comparable.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.preprocessing import StandardScaler

n = 500
rng = np.random.default_rng(0)

datasets = {
    "blobs (2 centros)": make_blobs(n_samples=n, centers=2, random_state=8)[0],
    "blobs (5 centros)": make_blobs(n_samples=n, centers=5, cluster_std=0.8, random_state=3)[0],
    "circles": make_circles(n_samples=n, factor=0.5, noise=0.05, random_state=0)[0],
    "moons": make_moons(n_samples=n, noise=0.05, random_state=0)[0],
    "uniforme": rng.uniform(-1, 1, size=(n, 2)),
}
datasets = {nombre: StandardScaler().fit_transform(X) for nombre, X in datasets.items()}

**Elección de $\varepsilon$: gráfico de $k$-distancias.** Para cada punto calculamos la distancia a su $\nu$-ésimo vecino más cercano (contándose a sí mismo, igual que en la definición de núcleo) y ordenamos esas distancias. Si $\varepsilon$ supera esa distancia, el punto es núcleo. Los puntos dentro de los clusters tienen distancias chicas, y los del ruido o de las zonas poco densas, grandes. Un $\varepsilon$ razonable está cerca del "codo" de la curva, donde empieza a crecer rápido.

In [ ]:
from sklearn.neighbors import NearestNeighbors

nu = 5

# Hiperparámetros elegidos para cada conjunto
hiper = {
    "blobs (2 centros)": {"K": 2, "eps": 0.40, "nu": nu},
    "blobs (5 centros)": {"K": 5, "eps": 0.15, "nu": nu},
    "circles": {"K": 2, "eps": 0.25, "nu": nu},
    "moons": {"K": 2, "eps": 0.25, "nu": nu},
    "uniforme": {"K": 3, "eps": 0.30, "nu": nu},
}

fig, axes = plt.subplots(1, len(datasets), figsize=(22, 3.8))
for ax, (nombre, X) in zip(axes, datasets.items()):
    # kneighbors sobre los mismos datos incluye al propio punto (distancia 0) como primer vecino
    dist, _ = NearestNeighbors(n_neighbors=nu).fit(X).kneighbors(X)
    ax.plot(np.sort(dist[:, -1]))
    ax.axhline(hiper[nombre]["eps"], color="r", ls="--", lw=1, label=f"ε = {hiper[nombre]['eps']}")
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("puntos ordenados")
    ax.legend()
axes[0].set_ylabel(f"distancia al {nu}° vecino")
plt.tight_layout()
plt.show()

La grilla se arma con una función genérica que recibe un diccionario de métodos. En la guía siguiente, para agregar la fila de GMM alcanza con sumar una entrada a `metodos`.

In [ ]:
def ajustar_kmeans(nombre, X):
    km = myKMeans(K=hiper[nombre]["K"], n_init=10, random_state=0).fit(X)
    return km.etiquetas, km.centroides, f"K = {km.K}"

def ajustar_dbscan(nombre, X):
    db = myDBSCAN(eps=hiper[nombre]["eps"], nu=hiper[nombre]["nu"])
    et = db.fit(X)
    return et, None, f"ε = {db.eps}, ν = {db.nu}: {db.n_clusters} clusters, {np.sum(et == -1)} ruido"

metodos = {
    "K-medias": ajustar_kmeans,
    "DBSCAN": ajustar_dbscan,
}

def grilla_clustering(datasets, metodos, tam=3.6):
    """
    Grilla de gráficos: un conjunto de datos por columna y un método por fila.

    Parámetros:
    - datasets: dict nombre -> X (n, 2)
    - metodos: dict nombre_metodo -> función (nombre_dataset, X) -> (etiquetas, centroides o None, texto)
    """
    fig, axes = plt.subplots(
        len(metodos),
        len(datasets),
        figsize=(tam * len(datasets), tam * len(metodos)),
        squeeze=False,
    )
    for f, (metodo, ajustar) in enumerate(metodos.items()):
        for col, (nombre, X) in enumerate(datasets.items()):
            etiquetas, centroides, texto = ajustar(nombre, X)
            ax = axes[f, col]
            graficar_clusters(X, etiquetas, ax, centroides, titulo=texto)
            ax.set_xlabel("")
            ax.set_ylabel(metodo if col == 0 else "", fontsize=13)
            if f == 0:
                ax.set_title(f"{nombre}\n{texto}", fontsize=10)
    plt.tight_layout()
    return fig

fig = grilla_clustering(datasets, metodos)
plt.show()

#### a) ¿Dónde funciona y dónde falla cada método?

#### b) ¿Qué método identifica observaciones que no pertenecen a ningún cluster?

#### c) Datos sin estructura de grupos

Para ver mejor qué hace cada método sobre el conjunto uniforme, variamos $K$ en K-medias y $\varepsilon$ en DBSCAN, y miramos también la curva del codo.

In [ ]:
X_u = datasets["uniforme"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))

for ax, K in zip(axes[0, :3], [2, 4, 8]):
    km = myKMeans(K=K, n_init=10, random_state=0).fit(X_u)
    graficar_clusters(X_u, km.etiquetas, ax, km.centroides, titulo=f"K-medias, K={K}")

J_u = [myKMeans(K=K, n_init=10, random_state=0).fit(X_u).J for K in Ks]
axes[0, 3].plot(Ks, J_u, "o-")
axes[0, 3].set_xticks(Ks)
axes[0, 3].set_xlabel("K")
axes[0, 3].set_ylabel("J")
axes[0, 3].set_title("Curva del codo (uniforme)")

for ax, eps in zip(axes[1], [0.15, 0.2, 0.3, 0.5]):
    db = myDBSCAN(eps=eps, nu=5)
    et = db.fit(X_u)
    graficar_clusters(X_u, et, ax, titulo=f"DBSCAN, ε={eps}: {db.n_clusters} clusters, {np.sum(et == -1)} ruido")

plt.tight_layout()
plt.show()

**Para discutir:** ¿Qué devuelve cada método cuando no hay grupos que descubrir?